# Lab 09 · Route each case: rule or council

**Prepared by Mustafa Assaf** · Self-study: Build your council further · 15–20 minute lab

## Send each case to the right handler

**So far:** Labs 06 and 07 built a council that debates, defers to the human, and remembers. One debate costs about five model calls, and many cases do not need one.

**New:** *routing*. A *router* reads a case and picks the handler that fits. A *dispatch table* maps each route to its handler:

- **rule:** the written policy, in plain code, with no model call.
- **council:** a full debate, as in Lab 06, for a case with a tension that the policy does not cover.

The router is a model call, because each case has a free-text note that the rule cannot read. Our code still owns the routes: a reply it cannot use falls back to the council, the careful path. Then we compare the router with the routes a human architect chose, and count the calls. The two possible mistakes cost different things: a routine case sent to the council wastes calls, but a contested case sent to the rule skips the debate it needed.

## Pattern · Routing

Routing is not a separate entry in the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. It combines two of the guide's ideas: the risk classification of *Human in the Loop* (classify, then send only high-risk cases to the expensive path) and the dispatch of *Registry & Adapter* (look up a handler and call it through one interface).

| | |
| --- | --- |
| **Context** | Cases differ in how much deliberation they need, and the most careful path is also the most expensive. |
| **Problem** | Sending every case to the council wastes calls; sending every case to a rule skips debates that were needed. |
| **Solution** | Classify each case, dispatch it to the handler that fits, and fall back to the careful path when the classification is unusable. |
| **Use when** | Cases fall into recognizable classes; handlers differ greatly in cost; one kind of wrong route is cheaper than the other. |
| **Trade-off** | Saves calls on routine cases, but the router adds its own calls, and its mistakes stay silent unless they are measured. |
| **Flow** | Case → Classify → Dispatch → Handler |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Cases, and the routes a human chose

Each case has the two facts that the policy reads, and a note that only a person or a model can weigh. `human_routes` holds the route a human architect chose for each case; the router never sees it. In practice mode, `scripted` holds the router's replies. It sends the Internal wiki to the council, which is a mistake.

In [ ]:
import json

cases = [  # the two facts the policy reads, plus a note that only a person or a model can weigh
    {"name": "Course portal", "team_size": 3, "independent_deploys": False, "note": "Two engineers run operations."},
    {"name": "Payments platform", "team_size": 12, "independent_deploys": True, "note": "One engineer runs operations."},
    {"name": "Internal wiki", "team_size": 9, "independent_deploys": False, "note": "Traffic is steady all year."},
    {"name": "Search platform", "team_size": 15, "independent_deploys": True, "note": "Four engineers run operations."},
    {"name": "Booking service", "team_size": 8, "independent_deploys": True, "note": "One engineer runs operations."},
    {"name": "Mobile backend", "team_size": 5, "independent_deploys": True, "note": "App review delays each release."},
]
human_routes = {"Course portal": "rule", "Payments platform": "council", "Internal wiki": "rule",
                "Search platform": "rule", "Booking service": "council", "Mobile backend": "rule"}
scripted = {  # practice mode: the router's replies as (route, reason)
    "Course portal": ("rule", "R1 gives a monolith, and the note agrees."),
    "Payments platform": ("council", "R1 says microservices, but one engineer cannot run them."),
    "Internal wiki": ("council", "Nine people is close to the threshold."),
    "Search platform": ("rule", "R1 says microservices, and four engineers can run them."),
    "Booking service": ("council", "R1 says microservices, but one engineer cannot run them."),
    "Mobile backend": ("rule", "Five people get a monolith, and the note does not change that.")}

## Step 2 · Two handlers and a dispatch table

`apply_rule` is the course policy from Lab 00. It reads only the two facts, so it gives the Payments platform microservices, although the note says that one engineer runs operations. `send_to_council` queues a case for a debate. In this lab it only records the case, and `COST` counts the five calls that a Lab 06 debate makes. `HANDLERS` is the dispatch table.

In [ ]:
POLICY = "R1: Choose microservices only when the team has 8+ people and services must deploy independently."
COST = {"rule": 0, "council": 5}  # model calls per case; the Lab 06 council made five
council_queue = []

def apply_rule(case):
    """The written policy in plain code. It reads team_size and independent_deploys, never the note."""
    return "microservices" if case["team_size"] >= 8 and case["independent_deploys"] else "modular monolith"

def send_to_council(case):
    """Queue the case for a full council debate, as in Lab 06."""
    council_queue.append(case["name"])
    return "queued for the council"

HANDLERS = {"rule": apply_rule, "council": send_to_council}
print(HANDLERS["rule"](cases[1]), "| but the note says:", cases[1]["note"])

## Step 3 · The router

The prompt holds the policy, three worked examples, the case, and when to choose each route: the council only if the note gives a reason against the policy's answer. The examples matter. In our live tests with `qwen3.6-35b`, a router without them sent both contested cases to the rule, even when the prompt named the risk; with the examples, it sent both to the council. As in Lab 01, a worked example shows the model where the line is. The reply is JSON with a route and a reason. Two guards keep our code in charge: a reply that is not JSON, and a route that is not in `HANDLERS`, both fall back to the council.

In [ ]:
EXAMPLES = ("Examples:\n"  # worked examples, as in Lab 01, show the router where the line is
            "10 people, independent deploys, one engineer runs operations -> council: one cannot run many services\n"
            "20 people, independent deploys, six engineers run operations -> rule: six can run the services\n"
            "4 people, one shared release, releases happen monthly -> rule: nothing argues against the policy")

def route(case):
    """Ask a model for a route. A reply our code cannot use falls back to the council, the careful path."""
    prompt = (f"{POLICY}\n{EXAMPLES}\nCase: {json.dumps(case)}\nRoute the case: council if the note gives a "
              "reason against the policy's answer; otherwise rule. "
              'Reply with JSON only, no code fences: {"route": "rule" or "council", "reason": "a short sentence"}')
    example = json.dumps(dict(zip(["route", "reason"], scripted[case["name"]])))
    try:
        reply = json.loads(ask(prompt, example=example))
    except json.JSONDecodeError:
        return "council", "the reply was not JSON"
    if reply.get("route") not in HANDLERS:
        return "council", f"unknown route {reply.get('route')!r}"
    return reply["route"], reply.get("reason", "")

## Step 4 · Route every case and count the calls

Each case is routed, then handled by `HANDLERS[chosen]`. The router itself costs one call per case, so it counts too. Each disagreement with the human is either a wasted debate or a skipped one. The assert stops the run if a contested case skipped the council, the costly kind of mistake.

In [ ]:
calls, wasted, skipped = len(cases), [], []  # the router costs one call per case
for case in cases:
    chosen, reason = route(case)
    result = HANDLERS[chosen](case)
    calls += COST[chosen]
    if chosen != human_routes[case["name"]]:
        (skipped if chosen == "rule" else wasted).append(case["name"])
    print(f"{case['name']:<18} -> {chosen:<8} {result:<23} {reason}")
agreed = len(cases) - len(wasted) - len(skipped)
print(f"\nThe router agrees with the human on {agreed} of {len(cases)} cases. Model calls: {calls}, "
      f"against {COST['council'] * len(cases)} if every case went to the council.")
print("Wasted debates:", wasted, "| skipped debates:", skipped)
assert not skipped, "A case that needed a debate skipped the council."

## What you should see

Step 2 prints `microservices | but the note says: One engineer runs operations.` In Step 4, the Payments platform and the Booking service go to the council, as the human chose. The Internal wiki goes there too, which wastes a debate. The router agrees with the human on 5 of 6 cases, and the run costs 21 model calls: 6 for routing and 15 for three debates. Sending every case to the council would cost 30.

The debates are counted here, not run: `council_queue` lists the cases that a Lab 06 council would now debate.

A router pays off only if it rarely wastes a debate. In three live runs with `qwen3.6-35b`, the router never skipped a needed debate, but it wasted between zero and three. With three wasted debates, the run cost 31 calls: more than sending every case to the council, because the router adds its own calls.

## Try it

1. Make the router send the Booking service to `"rule"`. The assert stops the run: a case at the threshold, with one operations engineer, skipped its debate.
2. Make one scripted route unusable, for example `"maybe"`. That case falls back to the council.
3. Change the Search platform's note to "One engineer runs operations." Which route should it get now? Update `human_routes` and its scripted reply.
4. In live mode, does the router agree with the human? Read its reason wherever it does not. Then remove `{EXAMPLES}` from the prompt and run Step 4 again. Which kind of mistake appears?

## Takeaway

Routing spends effort where it matters: plain code for routine cases, and the council for contested ones. The router is a model, but the routes, the fallback, and the check of its mistakes stay in code. Count the router's own calls, and weigh the two kinds of mistake differently. Lab 08 measured whether the council's extra calls buy better decisions; routing decides when to spend them.

## Read further

- [Building effective agents: routing](https://www.anthropic.com/research/building-effective-agents)